# �� 伴奏制作 worker（LessonMate）

**最简用法**：菜单 文件→上传笔记本 上传本文件，然后 代码执行程序→全部运行。

**用法**：菜单栏 `代码执行程序` → `全部运行`，然后挂着别关浏览器标签。

它会自动：装 AI 分离引擎（Demucs，UVR5 同款）→ 每 15 秒问你服务器有没有新任务 → 有就 GPU 分离 → 伴奏传回服务器。

老师们的使用入口：**http://134.175.44.202/banrace/**（不需要梯子）

---

In [ ]:
#@title 1️⃣ 安装引擎（约 2 分钟，只需每次会话跑一次）
SERVER = 'http://134.175.44.202/banrace'  #@param {type:"string"}
WORKER_KEY = 'banrace-worker-2026'  #@param {type:"string"}
!pip -q install demucs soundfile
import torch
print('GPU 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 型号:', torch.cuda.get_device_name(0))

In [ ]:
#@title 2️⃣ 启动 worker（保持运行，不要关）
import os, time, subprocess, requests, glob
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
H = {'X-Worker-Key': WORKER_KEY}

def claim():
    """返回 (音频字节或None, 任务id, 任务名)"""
    r = requests.post(SERVER + '/api/claim', headers=H, timeout=120)
    ct = r.headers.get('Content-Type', '')
    if 'json' in ct:
        return None, None, None
    # 二进制 = 领到任务，从任务列表反查 id/名字
    r2 = requests.get(SERVER + '/api/tasks', timeout=30)
    proc = [t for t in r2.json().get('tasks', []) if t['status'] == 'processing']
    tid = proc[0]['id'] if proc else 'unknown'
    name = proc[0].get('name', 'song.mp3') if proc else 'song.mp3'
    return r.content, tid, name

def separate(audio_bytes, name):
    inpath = os.path.join(WORK, 'in_' + name)
    with open(inpath, 'wb') as f:
        f.write(audio_bytes)
    outdir = os.path.join(WORK, 'out')
    subprocess.run(['rm', '-rf', outdir], check=False)
    cmd = ['python3', '-m', 'demucs', '--two-stems', 'vocals', '-n', 'htdemucs_ft',
           '--overlap', '0.5', '-o', outdir, inpath]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr[-800:])
    cands = glob.glob(outdir + '/**/no_vocals.wav', recursive=True)
    if not cands:
        raise RuntimeError('no_vocals.wav not found')
    wav = cands[0]
    mp3 = wav.replace('.wav', '.mp3')
    subprocess.run(['ffmpeg', '-y', '-i', wav, '-b:a', '320k', mp3],
                   capture_output=True, check=False)
    final = mp3 if os.path.exists(mp3) else wav
    with open(final, 'rb') as f:
        return f.read()

def complete(tid, mp3_bytes):
    r = requests.post(SERVER + '/api/complete', headers=H,
                      data={'id': tid},
                      files={'file': ('accompaniment.mp3', mp3_bytes)}, timeout=300)
    return r.json()

def fail(tid, err):
    try:
        requests.post(SERVER + '/api/fail', headers=H, json={'id': tid, 'error': err[:300]}, timeout=30)
    except Exception:
        pass

print('worker 启动，每 15 秒检查新任务 …')
n_done = 0
while True:
    try:
        audio, tid, name = claim()
        if audio:
            print('⚡ 开始处理:', name, tid)
            try:
                acc = separate(audio, name)
                complete(tid, acc)
                n_done += 1
                print('✅ 完成:', name, '（累计', n_done, '首）')
            except Exception as e:
                print('❌ 失败:', e)
                fail(tid, str(e))
        else:
            time.sleep(15)
    except KeyboardInterrupt:
        print('worker 手动停止')
        break
    except Exception as e:
        print('网络波动，10 秒后重试:', e)
        time.sleep(10)
